In [3]:
!pip install -r requirements.txt

  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 628.9 kB/s eta 0:00:0000:0100:01
Using cached threadpoolctl-3.5.0-py3-none-any.whl (18 kB)
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 2.2.0
    Uninstalling threadpoolctl-2.2.0:
      Successfully uninstalled threadpoolctl-2.2.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2


In [4]:
!conda list | grep scikit-learn

scikit-learn              1.5.0                    pypi_0    pypi


In [5]:
import pickle
import pandas as pd
import uuid

In [59]:
year = 2023
month = 3
taxi_type = 'yellow'
categorical = ['PULocationID', 'DOLocationID']
output_file = f'output/{taxi_type}_{year:04d}-{month:02d}.parquet'
input_file = './data/yellow_tripdata_2023-03.parquet'
# input_file = f'https://s3.amazonaws.com/nyc-tlc/trip+data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet'

In [39]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [37]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [62]:
def apply_model():
    df = read_data(input_file)
    
    # #Make Prediction
    dicts = df[categorical].to_dict(orient='records')
    X_val = dv.transform(dicts)
    y_pred = model.predict(X_val)

    # Q1 The standard deviation of the predicted duration for this dataset.
    sd = y_pred.std()
    print('sd of y_pred : ', sd)

    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

    # Q2 Wrtting the ride_id and predicitons in a new data frame
    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['predicted_duration'] = y_pred
    df_result.to_parquet(
        output_file,
        engine='pyarrow',
        compression=None,
        index=False
    )
    

In [63]:
apply_model()

sd of y_pred :  6.247488852238703


In [65]:
# # Q2 Printing the size of the df_result file
!ls -lh output/

total 163848
-rw-r--r--@ 1 dushi  staff    65M Jun 20 00:32 yellow_2023-03.parquet


In [67]:
#Q3 Creating the scoring script
# Which command you need to execute for that?
!jupyter nbconvert --to script starter.ipynb


[NbConvertApp] Converting notebook starter.ipynb to script
[NbConvertApp] Writing 2067 bytes to starter.py
